In [1]:
import pathlib
from datetime import datetime, timedelta
from functools import partial

import numpy as np
import pandas as pd
import xarray as xr
import torch

from matplotlib import pyplot as plt
from torchvision.ops import MLP
from torchvision.utils import _log_api_usage_once
from lightning.pytorch import LightningModule

In [2]:
data_path = pathlib.Path("../data/cloudsat-goes-paired/")
files = sorted(list(data_path.glob("*.nc")))
len(files)

2000

In [3]:
from collections import OrderedDict
from torchvision.models.vision_transformer import EncoderBlock

class Encoder(torch.nn.Module):
    """Transformer Model Encoder for sequence to sequence translation."""

    def __init__(
        self,
        seq_length: int,
        num_layers: int,
        num_heads: int,
        hidden_dim: int,
        mlp_dim: int,
        dropout: float,
        attention_dropout: float,
        norm_layer: Callable[..., torch.nn.Module] = partial(torch.nn.LayerNorm, eps=1e-6),
    ):
        super().__init__()
        # Note that batch_size is on the first dim because
        # we have batch_first=True in nn.MultiAttention() by default
        self.dropout = torch.nn.Dropout(dropout)
        layers: OrderedDict[str, torch.nn.Module] = OrderedDict()
        for i in range(num_layers):
            layers[f"encoder_layer_{i}"] = EncoderBlock(
                num_heads,
                hidden_dim,
                mlp_dim,
                dropout,
                attention_dropout,
                norm_layer,
            )
        self.layers = torch.nn.Sequential(layers)
        self.ln = norm_layer(hidden_dim)

    def forward(self, input: torch.Tensor):
        torch._assert(input.dim() == 3, f"Expected (batch_size, seq_length, hidden_dim) got {input.shape}")
        return self.ln(self.layers(self.dropout(input)))

In [67]:
class CloudsatTransformer(LightningModule):
    """
    A Transformer model to reconstruct CloudSat observations
    """
    def __init__(
        self,
        image_size: int,
        image_channels: int,
        geoloc_channels: int, 
        output_channels: int,
        patch_size: int,
        encoder_layers: int,
        decoder_layers: int, 
        num_heads: int,
        hidden_dim: int,
        mlp_dim: int,
        dropout: float = 0.0,
        attention_dropout: float = 0.0,
        num_classes: int = 1000,
        representation_size: Optional[int] = None,
        norm_layer = partial(torch.nn.LayerNorm, eps=1e-6),
    ):
        super().__init__()
        _log_api_usage_once(self)
        torch._assert(image_size % patch_size == 0, "Input shape indivisible by patch size!")
        self.image_size = image_size
        self.image_channels = image_channels
        self.geoloc_channels = geoloc_channels
        self.output_channels = output_channels
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size)**2
        self.hidden_dim = hidden_dim
        self.mlp_dim = mlp_dim
        self.attention_dropout = attention_dropout
        self.dropout = dropout
        self.num_classes = num_classes
        self.representation_size = representation_size

        # Add a class token
        self.class_token = torch.nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.seq_length = self.num_patches + 1

        self.encoder = Encoder(
            self.seq_length,
            encoder_layers,
            num_heads,
            hidden_dim,
            mlp_dim,
            dropout,
            attention_dropout,
            norm_layer,
        )

        self.decoder = Encoder(
            self.seq_length+256-1, 
            decoder_layers, 
            num_heads, 
            hidden_dim,
            mlp_dim,
            dropout,
            attention_dropout,
            norm_layer,
        )

        self.image_encoder = image_conv = torch.nn.Sequential(
            torch.nn.Conv2d(
                self.image_channels, 
                self.hidden_dim, 
                kernel_size=self.patch_size, 
                stride=self.patch_size
            ), 
            torch.nn.Sigmoid()
        )

        self.geopos_encoder = MLP(
            self.geoloc_channels, 
            [self.hidden_dim], 
            activation_layer=torch.nn.Sigmoid(), 
        )

        self.geopos_pool = torch.nn.AdaptiveAvgPool2d(image_size // patch_size)

        self.output_head = MLP(
            self.hidden_dim, 
            [self.hidden_dim, self.hidden_dim, self.hidden_dim, self.output_channels], 
            activation_layer=torch.nn.LeakyReLU, 
        )

    def encode_image(self, x):
        n = x.shape[0]
        return self.image_encoder(image_input).reshape(n, self.hidden_dim, -1).permute(0, 2, 1)
        
    def encode_geopos(self, geopos):
        n = geopos.shape[0]
        return self.geopos_encoder(
            self.geopos_pool(geopos).permute(0,2,3,1).reshape(-1, self.geoloc_channels)
        ).reshape(n, -1, self.hidden_dim)

    def forward_encode(self, x, geopos):
        n = x.shape[0]
        x = self.encode_image(x) + self.encode_geopos(geopos)
        # shuffle x
        input_shuffle = torch.randperm(x.shape[1])
        x = torch.cat(
            [
                self.class_token.expand(n, -1, -1),
                x[:, input_shuffle], 
            ],
            1
        )

        return self.encoder(x)

    def forward_decode(self, x, geopos):
        n_batch, n_target = geopos.shape[0:2]
        
        output_shuffle = torch.randperm(n_target)
        output_unshuffle = torch.sort(output_shuffle).indices

        geopos_x = self.geopos_encoder(
            geopos.reshape(-1, self.geoloc_channels)
        ).reshape(n_batch, n_target, self.hidden_dim)
        
        x = torch.cat([
            x[:,1:],
            geopos_x[:, output_shuffle], 
        ], 1)

        x = self.decoder(x)
    
        x = x[:, -n_target:][:, output_unshuffle]

        return x

    def forward_output(x):
        n_batch = x.shape[0]

        x = self.output_head(x.reshape(-1, self.hidden_dim))

        return x.reshape(n_batch, -1, self.output_channels)
        

    def forward(self, x, image_geopos, target_geopos):
        x = self.forward_encode(x, image_geopos)

        x = self.forward_decode(x, target_geopos)

        x = self.output_head(x)

        return x

In [68]:
model = CloudsatTransformer(
    image_size=256,
    image_channels=16,
    geoloc_channels=13,
    output_channels=125,
    patch_size=8,
    encoder_layers=16,
    decoder_layers=8,
    num_heads=12,
    hidden_dim=1536,
    mlp_dim=1536*4,
)

### Test with an example file

In [6]:
dt = xr.open_datatree(files[0])

In [7]:
image_input = dt.geo_patch.data
image_input[:6] = image_input[:6] / 100
image_input[6:] = image_input[6:] / 140 - 180
image_input = np.clip(image_input, 0, 1)
image_input = torch.tensor(image_input.data[np.newaxis])

In [8]:
time_of_day = (dt.geo_patch.t.values - dt.geo_patch.t.values.astype("datetime64[D]")) / np.timedelta64(1, "D").astype("timedelta64[ns]") * 2 * np.pi
time_of_year = (dt.geo_patch.t.values - dt.geo_patch.t.values.astype("datetime64[Y]")) / np.timedelta64(1, "Y").astype("timedelta64[ns]") * 2 * np.pi

geoloc_input = torch.tensor(np.stack([
    np.sin(np.radians(dt.geo_patch.longitude)), 
    np.cos(np.radians(dt.geo_patch.longitude)), 
    np.sin(np.radians(dt.geo_patch.latitude)), 
    np.sin(np.radians(dt.geo_patch.sat_angle[1])), 
    np.cos(np.radians(dt.geo_patch.sat_angle[1])), 
    np.sin(np.radians(dt.geo_patch.sat_angle[0])), 
    np.sin(np.radians(dt.geo_patch.solar_angle[1])), 
    np.cos(np.radians(dt.geo_patch.solar_angle[1])), 
    np.sin(np.radians(dt.geo_patch.solar_angle[0])), 
    np.full(dt.geo_patch.longitude.shape, np.sin(time_of_day), np.float32), 
    np.full(dt.geo_patch.longitude.shape, np.cos(time_of_day), np.float32), 
    np.full(dt.geo_patch.longitude.shape, np.sin(time_of_year), np.float32), 
    np.full(dt.geo_patch.longitude.shape, np.cos(time_of_year), np.float32), 
], 0).astype(np.float32)[np.newaxis])

In [9]:
def get_sza_and_azi(date: datetime, lat: float, lon: float) -> tuple[float, float]:
    """Get the solar zenith angle at a specific time/lat/lon

    Parameters
    ----------
    date : datetime | array of datetime like
        Dates of the points
    lat : float
        Latitudes
    lon : float
        Longitudes

    Returns
    -------
    sza: float
        The solar zenith angle in degrees, where 0 is directly above, 90 is on 
        the horizon and 180 is directly below
    saa: float
        The solar azimuth angle in degrees, clockwise from North
    """
    try:
        date = pd.DatetimeIndex(date)
    except TypeError:
        date = pd.DatetimeIndex([date])
    day_of_year = date.dayofyear.to_numpy()
    hour_of_day = (
        date.hour + date.minute / 60 + date.second / 60 /60
    ).to_numpy()

    # calculate approx time equation as angle for 365 day year
    equation_of_time_approx = 2.0 * np.pi * day_of_year / 365.0

    # calculate the solar declination for the given day
    # the declination varies due to the fact that the earth rotation axis
    # is not perpendicular to the ecliptic plane
    solar_declination = (
        0.006918
        - 0.399912 * np.cos(equation_of_time_approx)
        - 0.006758 * np.cos(2.0 * equation_of_time_approx)
        - 0.002697 * np.cos(3.0 * equation_of_time_approx)
        + 0.070257 * np.sin(equation_of_time_approx)
        + 0.000907 * np.sin(2.0 * equation_of_time_approx)
        + 0.001480 * np.sin(3.0 * equation_of_time_approx)
    )

    # equation of time, used to compensate for the earth's elliptical orbit
    # around the sun and its axial tilt when calculating solar time
    # eqt is the correction in hours
    equation_of_time = 2.0 * np.pi * day_of_year / 366.0
    equation_of_time = (
        0.0072 * np.cos(equation_of_time)
        - 0.0528 * np.cos(2.0 * equation_of_time)
        - 0.0012 * np.cos(3.0 * equation_of_time)
        - 0.1229 * np.sin(equation_of_time)
        - 0.1565 * np.sin(2.0 * equation_of_time)
        - 0.0041 * np.sin(3.0 * equation_of_time)
    )

    # calculate the solar zenith angle
    omega = np.radians(
        (360.0 / 24.0) * (hour_of_day + lon / 15.0 + equation_of_time - 12.0)
    )
    sunh = np.sin(solar_declination) * np.sin(np.radians(lat)) + np.cos(
        solar_declination
    ) * np.cos(np.radians(lat)) * np.cos(omega)

    solar_elevation = np.arcsin(np.clip(sunh, -1, 1))
    solar_zenith_angle = np.pi / 2.0 - solar_elevation

    # Solar azimuth added by yaswant
    azimuth = (
        np.sin(solar_declination) * np.cos(np.radians(lat))
        - np.cos(solar_declination) * np.sin(np.radians(lat)) * np.cos(omega)
    ) / np.cos(np.pi / 2.0 - solar_zenith_angle)

    solar_azimuth_angle = np.arccos(np.clip(azimuth, -1, 1))

    return np.degrees(solar_zenith_angle), np.degrees(solar_azimuth_angle)

In [10]:
cloudsat_offset = (dt.cloudsat_unaligned.Nray.size-256)//2
cloudsat_unaligned = dt.cloudsat_unaligned.isel(Nray=slice(cloudsat_offset, cloudsat_offset+256))

cloudsat_sza, cloudsat_azi = get_sza_and_azi(
    cloudsat_unaligned.Profile_time.values,
    cloudsat_unaligned.Latitude.values, 
    cloudsat_unaligned.Longitude.values,
)

time_of_day = (cloudsat_unaligned.Profile_time.values - cloudsat_unaligned.Profile_time.values.astype("datetime64[D]")) / np.timedelta64(1, "D").astype("timedelta64[ns]") * 2 * np.pi
time_of_year = (cloudsat_unaligned.Profile_time.values - cloudsat_unaligned.Profile_time.astype("datetime64[Y]")) / np.timedelta64(1, "Y").astype("timedelta64[ns]") * 2 * np.pi

geoloc_output = torch.tensor(np.stack([
    np.sin(np.radians(cloudsat_unaligned.Longitude)), 
    np.cos(np.radians(cloudsat_unaligned.Longitude)), 
    np.sin(np.radians(cloudsat_unaligned.Latitude)), 
    np.full(cloudsat_unaligned.Longitude.shape, np.sin(0), np.float32), # 0 degree sat angle for all
    np.full(cloudsat_unaligned.Longitude.shape, np.cos(0), np.float32), # 0 degree sat angle for all
    np.full(cloudsat_unaligned.Longitude.shape, np.sin(0), np.float32), # 0 degree sat angle for all
    np.sin(np.radians(cloudsat_azi)), # need to calc solar angles
    np.cos(np.radians(cloudsat_azi)), # need to calc solar angles
    np.sin(np.radians(cloudsat_sza)), # need to calc solar angles
    np.full(cloudsat_unaligned.Longitude.shape, np.sin(time_of_day), np.float32), 
    np.full(cloudsat_unaligned.Longitude.shape, np.cos(time_of_day), np.float32), 
    np.full(cloudsat_unaligned.Longitude.shape, np.sin(time_of_year), np.float32), 
    np.full(cloudsat_unaligned.Longitude.shape, np.cos(time_of_year), np.float32), 
], -1).astype(np.float32)[np.newaxis])


In [11]:
image_input.shape, geoloc_input.shape, geoloc_output.shape

(torch.Size([1, 16, 256, 256]),
 torch.Size([1, 13, 256, 256]),
 torch.Size([1, 256, 13]))

In [56]:
geoloc_output.shape

torch.Size([1, 256, 13])

In [69]:
model.forward(image_input, geoloc_input, geoloc_output).shape

torch.Size([1, 256, 125])